# 03 - Interpret Model Output Samples

Neat model and graph execution can return more than one output shape. Sometimes you receive a tensor list. Sometimes you receive a `Sample` whose tensor data is stored in `sample.tensor`, `sample.tensors`, or nested `sample.fields`.

This notebook uses a real image and a pass-through graph so the output payload is easy to verify while focusing on the output container API.

## 1. Import Packages

In [1]:
from pathlib import Path
import numpy as np
import cv2
import pyneat


## 2. Load Image

In [2]:
IMAGE_PATH = Path("../assets/images/image.png")
image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
print("image:", IMAGE_PATH, image_bgr.shape, image_bgr.dtype)


image: ../assets/images/image.png (563, 1000, 3) uint8


## 3. Create An RGB Tensor

In [3]:
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
tensor = pyneat.Tensor.from_numpy(
    image_rgb,
    copy=True,
    image_format=pyneat.PixelFormat.RGB,
)
print("tensor:", tuple(tensor.shape), tensor.dtype)


tensor: (563, 1000, 3) TensorDType.UInt8


## 4. Build A Pass-Through Graph

The graph has no model. It only moves the image from input to output, which lets us study output structure without model-specific decoding.

In [4]:
graph_options = pyneat.GraphOptions()
graph_options.verbose = pyneat.VerboseOptions.quiet()

height, width = image_rgb.shape[:2]
input_options = pyneat.InputOptions()
input_options.format = pyneat.Format.RGB
input_options.width = width
input_options.height = height
input_options.depth = 3

graph = pyneat.Graph("output_sample_reader", graph_options)
graph.add(pyneat.nodes.input(input_options))
graph.add(pyneat.nodes.output())
print(graph.describe())


0) Input  [mysrc]
1) Output


## 5. Tensor Input Returns A Tensor List

When the input is a list of tensors, `graph.run([tensor])` returns a tensor list. This is the simplest output shape.

In [5]:
tensor_outputs = graph.run([tensor])

print("python type:", type(tensor_outputs))
print("output count:", len(tensor_outputs))
print("first output type:", type(tensor_outputs[0]))
print("first output shape:", tuple(tensor_outputs[0].shape))

arr = tensor_outputs[0].to_numpy(copy=True)
print("same pixels:", bool(np.array_equal(image_rgb, arr)))


python type: <class 'list'>
output count: 1
first output type: <class 'pyneat._pyneat_core.Tensor'>
first output shape: (563, 1000, 3)
same pixels: True


## 6. Sample Input Returns A Sample

When the input is a list of `Sample` objects, `graph.run([sample])` returns a `Sample`. This is the API surface you should understand for model outputs, named graph outputs, streams, and metadata-aware pipelines.

In [6]:
input_sample = pyneat.Sample()
input_sample.kind = pyneat.SampleKind.Tensor
input_sample.tensor = tensor
input_sample.stream_id = "tutorial"
input_sample.frame_id = 1
input_sample.pts_ns = 0

sample_output = graph.run([input_sample])

print("python type:", type(sample_output))
print("sample kind:", sample_output.kind)
print("has sample.tensor:", sample_output.tensor is not None)
print("sample.tensors:", len(sample_output.tensors))
print("sample.fields:", len(sample_output.fields))
print("stream/frame:", sample_output.stream_id, sample_output.frame_id)


python type: <class 'pyneat._pyneat_core.Sample'>
sample kind: SampleKind.TensorSet
has sample.tensor: False
sample.tensors: 1
sample.fields: 0
stream/frame: tutorial 1


## 7. Read Tensor Data Defensively

Do not assume every output has `sample.tensor`. Some outputs use `sample.tensors`; bundle-style outputs use `sample.fields`. Read the structure first, then touch the payload.

In [7]:
def tensors_from_sample(sample):
    tensors = []

    if sample is None:
        return tensors

    if sample.tensor is not None:
        tensors.append(sample.tensor)

    if len(sample.tensors) > 0:
        tensors.extend(list(sample.tensors))

    for field in sample.fields:
        if field.tensor is not None:
            tensors.append(field.tensor)
        if len(field.tensors) > 0:
            tensors.extend(list(field.tensors))

    return tensors


def first_tensor(sample):
    tensors = tensors_from_sample(sample)
    if not tensors:
        raise RuntimeError("sample does not contain tensor output")
    return tensors[0]

out_tensor = first_tensor(sample_output)
out = out_tensor.to_numpy(copy=True)
print("output tensor shape:", tuple(out_tensor.shape))
print("same pixels:", bool(np.array_equal(image_rgb, out)))


output tensor shape: (563, 1000, 3)
same pixels: True


## 8. Validate Output Before Using It

A robust consumer checks that the output exists, has the expected rank, and has the dtype/layout it expects before doing downstream work.

In [8]:
def validate_image_tensor(tensor, expected_height, expected_width, expected_channels=3):
    shape = tuple(tensor.shape)
    if len(shape) != 3:
        raise RuntimeError(f"expected HWC image tensor rank 3, got shape {shape}")
    if shape[:2] != (expected_height, expected_width):
        raise RuntimeError(f"expected image size {(expected_height, expected_width)}, got {shape[:2]}")
    if shape[2] != expected_channels:
        raise RuntimeError(f"expected {expected_channels} channels, got {shape[2]}")
    return True

print("valid image tensor:", validate_image_tensor(out_tensor, height, width))
print("dtype:", out_tensor.dtype)
print("layout:", out_tensor.layout)
print("semantic:", out_tensor.semantic)


valid image tensor: True
dtype: TensorDType.UInt8
layout: TensorLayout.HWC
semantic: <pyneat._pyneat_core.Semantic object at 0xffff20af7e70>


## 9. Simulate Common Output Shapes

These small samples show why the helper checks all three places: `tensor`, `tensors`, and `fields`.

In [9]:
single_tensor_sample = pyneat.Sample()
single_tensor_sample.kind = pyneat.SampleKind.Tensor
single_tensor_sample.tensor = tensor

tensor_set_sample = pyneat.Sample()
tensor_set_sample.kind = pyneat.SampleKind.TensorSet
tensor_set_sample.tensors = [tensor]

field_sample = pyneat.Sample()
field_sample.kind = pyneat.SampleKind.Tensor
field_sample.tensor = tensor
field_sample.port_name = "image"

bundle_sample = pyneat.Sample()
bundle_sample.kind = pyneat.SampleKind.Bundle
bundle_sample.fields = [field_sample]

for label, sample in [
    ("single", single_tensor_sample),
    ("tensor_set", tensor_set_sample),
    ("bundle", bundle_sample),
]:
    tensors = tensors_from_sample(sample)
    print(label, "kind:", sample.kind, "tensor_count:", len(tensors), "first_shape:", tuple(tensors[0].shape))


single kind: SampleKind.Tensor tensor_count: 1 first_shape: (563, 1000, 3)
tensor_set kind: SampleKind.TensorSet tensor_count: 1 first_shape: (563, 1000, 3)
bundle kind: SampleKind.Bundle tensor_count: 1 first_shape: (563, 1000, 3)


## 10. Named Push/Pull Output

Named graph outputs are also read as samples. This mirrors camera, RTSP, multi-stream, and custom graph pipelines.

In [10]:
named_graph = pyneat.Graph("named_output_sample_reader", graph_options)
named_graph.add(pyneat.nodes.input("image"))
named_graph.add(pyneat.nodes.output("out"))
named_graph.connect("image", "out")

run = named_graph.build()
try:
    if not run.push("image", [input_sample]):
        raise RuntimeError(f"push failed: {run.last_error()}")
    pulled = run.pull("out", 2000)
finally:
    run.close()

print("pulled kind:", pulled.kind)
print("has tensor:", pulled.tensor is not None)
print("tensor set size:", len(pulled.tensors))
print("fields:", len(pulled.fields))
print("stream/frame:", pulled.stream_id, pulled.frame_id)

pulled_tensor = first_tensor(pulled)
pulled_array = pulled_tensor.to_numpy(copy=True)
print("same pixels:", bool(np.array_equal(image_rgb, pulled_array)))


pulled kind: SampleKind.TensorSet
has tensor: False
tensor set size: 1
fields: 0
stream/frame: tutorial 1
same pixels: True


## 11. What To Remember

If you pass tensors into `graph.run(...)`, expect a tensor list.

If you pass samples into `graph.run(...)`, expect a `Sample`.

For `Sample` outputs, classify before reading. Check `sample.tensor`, `sample.tensors`, and `sample.fields` instead of assuming one fixed layout.

For named push/pull pipelines, use the same defensive sample-reading helper.

## References

- Public tutorial: [interpret model output](https://developer.sima.ai/software/tutorials/interpret-model-output).
- Core source: [interpret_model_output.py](https://github.com/sima-neat/core/blob/main/tutorials/011_interpret_model_output/interpret_model_output.py).
- Custom graph push/pull sample pattern: [build_a_custom_data_graph.py](https://github.com/sima-neat/core/blob/main/tutorials/013_build_a_custom_data_graph/build_a_custom_data_graph.py).